# Stage C — end-to-end funnel on full papers (SAL)

The per-stage notebooks test each stage in isolation. **This notebook runs the whole funnel on real
papers** and compares every Stage B model end-to-end:

```
full paper  →  Stage A (section-aware keep)  →  Stage B (detector)  →  Stage C (LLM filter)  →  gaps
```

scored against each paper's **exhaustive** gold limitations. It also diagnoses *why* Stage C precision
looks capped (spoiler: the gold under-labels), grounded in the SAL annotation guideline.

**Dataset: SAL / RCT (Lan et al. 2024).** The only set we have with **full papers + exhaustive
sentence-level limitation labels + true negatives** — so it's the only one that supports an end-to-end
funnel evaluation (FWS / Sci-Sentence ship pre-sliced candidate sentences and can test *only* Stage B).
Test = **40 papers, 7,968 sentences, 205 gold limitations** (median 224 sents/paper, 2.6% prevalence).

## 1. Reconstruct the structured full papers (Stage A needs section titles)

The SAL CSV gives every sentence + label but **no section structure**. We recover each sentence's
section from the **PubMed Central BioC API** (cached). No container, no GROBID — just the section tag per
passage. (On raw PDFs we'd use `pdf_to_blocks` from the Stage A package instead.)

In [ ]:
import json, csv, io, urllib.request, time
from pathlib import Path
from collections import Counter

RAW  = "https://raw.githubusercontent.com/MengfeiLan/SAL_Type_Classification/main/data/test.csv"
BIOC = "https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/%s/unicode"
CACHE = Path("bioc_cache"); CACHE.mkdir(exist_ok=True)
MATCH_PREFIX = 50

def bioc_sections(pmcid):
    pid = pmcid if str(pmcid).upper().startswith("PMC") else "PMC"+str(pmcid)
    cf = CACHE / f"{pid}.json"
    if cf.exists():
        doc = json.loads(cf.read_text(encoding="utf-8"))
    else:
        doc = json.load(urllib.request.urlopen(BIOC % pid, timeout=90))
        cf.write_text(json.dumps(doc), encoding="utf-8")
    d = doc[0]["documents"][0]
    return [(p["infons"].get("section_type","?"), p.get("text","")) for p in d["passages"]]

def section_of(sentence, passages):
    key = " ".join(sentence.split())[:MATCH_PREFIX]
    for sect, text in passages:
        if key and key in " ".join(text.split()):
            return sect
    return "?"

# (Run to rebuild the per-sentence section map for the 40 test papers; cached after first run.)

**Result** — section distribution over the 7,968 test sentences, and the Stage-A ceiling:

```
METHODS 2324 · DISCUSS 1785 · RESULTS 1414 · INTRO 708 · ABSTRACT 602 · '?' 430 · CONCL 100 · ...
```

**203 of 205 gold limitations (99.0%) fall in DISCUSS + CONCL.** So keeping just those two sections is a
**0.99 recall ceiling** for Stage A on real papers — confirming the section-aware approach (and matching
what Kilicoglu's rule-based SOTA exploits).

## 2. Stage A — section-aware keep

Keep sentences whose recovered section is **DISCUSS or CONCL**; drop the rest.

| | kept | precision | recall | drop |
|---|---|---|---|---|
| Stage A only | 1885 / 7968 | 0.108 | **0.990** | 76% |

Prevalence rises **2.6% → 10.8%** — that higher prevalence is the whole point: it makes Stage B/C far
more precise than on the raw stream (where Stage B alone is only P≈0.17).

## 3. Stage B — all 8 models on the Stage-A slice (end-to-end)

Each detector trained on SAL-train (frozen encoder + logreg, or full fine-tune, dev-threshold-tuned),
run on the Stage-A slice, scored against the 205 gold limitations. Embeddings/fine-tunes were computed on
an H100 (30k sentences embed in ~11s). **A+B** = Stage A then Stage B.

| Stage B model | A+B  P / R / F1 |
|---|---|
| **PubMedBERT-FT** | 0.460 / 0.878 / **0.604** |
| SciBERT-FT | 0.405 / 0.873 / 0.553 |
| DistilBERT-FT | 0.377 / 0.849 / 0.523 |
| e5-base (frozen) | 0.312 / 0.849 / 0.457 |
| bge-base (frozen) | 0.299 / 0.815 / 0.437 |
| mpnet (frozen) | 0.304 / 0.776 / 0.437 |
| bge-small (frozen) | 0.298 / 0.810 / 0.436 |
| MiniLM-L6 (frozen) | 0.249 / 0.727 / 0.371 |

**Finding:** in-domain (biomedical SAL), **fine-tuned detectors win** — PubMedBERT best. This is the
*opposite* of the arXiv-gold result (where frozen bge beat the fine-tunes, which overfit out-of-domain).
Honest conclusion: **fine-tuned wins in-domain, frozen generalizes better cross-domain.**

## 4. Stage C — the LLM precision filter (yandexgpt, batched)

Stage C judges each Stage-B survivor ("is this the authors' own limitation?") and drops the No's.
Judged **once** over the union of all models' positives (the verdict is detector-independent). **A+B+C**:

| Stage B model | A+B F1 | A+B+C  P / R / **F1** |
|---|---|---|
| **PubMedBERT-FT** | 0.604 | 0.610 / 0.824 / **0.701** |
| DistilBERT-FT | 0.523 | 0.614 / 0.790 / 0.691 |
| SciBERT-FT | 0.553 | 0.595 / 0.810 / 0.686 |
| e5-base (frozen) | 0.457 | 0.549 / 0.790 / 0.648 |
| mpnet (frozen) | 0.437 | 0.553 / 0.712 / 0.623 |
| bge-base (frozen) | 0.437 | 0.523 / 0.761 / 0.620 |
| bge-small (frozen) | 0.436 | 0.530 / 0.737 / 0.616 |
| MiniLM-L6 (frozen) | 0.371 | 0.511 / 0.659 / 0.576 |

**Stage C lifts every model** — precision roughly doubles (~0.30 → ~0.60), +0.10–0.20 F1. Best full funnel
= **PubMedBERT A+B+C 0.701**.

In [ ]:
# The end-to-end scorer (given per-sentence {sec, label, pred} rows): recall/precision vs exhaustive gold.
def score(rows, flags):
    kept = [r for r, k in zip(rows, flags) if k]
    tp = sum(r["label"] for r in kept)
    GOLD = sum(r["label"] for r in rows)
    R = tp / GOLD
    P = tp / len(kept) if kept else 0.0
    F = 2*P*R/(P+R) if P+R else 0.0
    return round(P,3), round(R,3), round(F,3)

# Stage A mask = section in {DISCUSS, CONCL};  A+B = inA & detector;  A+B+C = A+B & stage_c_verdict

## 5. Why isn't Stage C precision ~1.0? — it's the **gold**, not the model

Stage C looks "only" ~0.60 precision. We diagnosed this with a labeled sample of the actual sentences it
judges (PubMedBERT A+B positives: 60 gold-limitations + 60 hard negatives):

- **Model/batch/prompt sweep:** batch 20 is the sweet spot (batching doesn't hurt); **±1 context** best;
  a best-practice prompt reaches **F1 0.827** vs gold. But **precision plateaus ~0.70–0.75 in every config.**
- **Manual adjudication:** I read 5 "false positives" *in their papers* — **4 of 5 are genuine limitations
  SAL never tagged** (e.g. *"the risk of type 1 error was considerable"*, *"this could have introduced a
  bias"*, *"the choice of … may have been too ambitious"*).

So the LLM is **correctly** flagging limitations; it's scored wrong only because SAL didn't annotate them.

### The SAL annotation criteria explain it exactly

From SAL's `annotation_guideline_limitation_types.pdf`:
1. **Only sentences pre-marked as limitation sentences get labeled** — limitations in Results/Interpretation
   subsections are never tagged.
2. **Minimality** — they tag the shortest *clause*, not the sentence.
3. **Consequences/inferences are EXCLUDED** (direct quote): tag *"the lack of a no treatment arm"*, **not**
   the *"means we cannot compare…"* consequence.
4. Must fit one of **16 explicit categories**, type explicit in text.

So SAL tags the **minimal cause-clause of a typed limitation**; our funnel flags **any sentence expressing a
limitation**. The gap is a **task-definition mismatch by design** — precision vs SAL is a *lower bound*.
Confirmed systemic: **Kilicoglu-2018** (same group, same scheme) shows the same pattern.

## 6. Stage C aligned to SAL's criteria (cause-not-consequence)

We rewrote the Stage C prompt to match SAL's rules (tag the cause, drop consequences/inferences, 16
categories). Re-running the full funnel:

| Stage B model | A+B+C (SAL-criteria)  P / R / **F1** | *A+B+C (broad)* F1 |
|---|---|---|
| **PubMedBERT-FT** | 0.662 / 0.498 / **0.568** | *0.701* |
| SciBERT-FT | 0.646 / 0.498 / 0.562 | *0.686* |
| DistilBERT-FT | 0.645 / 0.488 / 0.556 | *0.691* |
| e5-base (frozen) | 0.568 / 0.507 / 0.536 | *0.648* |
| bge-small (frozen) | 0.588 / 0.488 / 0.533 | *0.616* |
| MiniLM-L6 (frozen) | 0.563 / 0.434 / 0.490 | *0.576* |

**What it shows:** aligning to SAL raises precision (0.610 → 0.662) but **recall collapses** (0.824 → 0.498),
so F1 falls (**0.701 → 0.568**). This **quantifies the definition gap**: matching SAL exactly costs ~0.13 F1,
almost all recall — because SAL's definition is *narrower* than "a limitation of the study." The broad judge
wasn't wrong; it answers a broader (and more useful) question than SAL annotates.

## 7. Cross-check — Kilicoglu-2018 (same input/output structure)

Independent corpus (clinical research, not just RCT), same structure (full papers, per-sentence limitation
labels + negatives), TEST = 1,505 sentences at 31.8% prevalence.

| stage | P | R | F1 |
|---|---|---|---|
| Stage B (bge+logreg) | 0.719 | 0.781 | 0.749 |
| + Stage C | 0.876 | 0.503 | 0.639 |

The **same gold under-labeling recurs** — the Stage C "false positives" are again real limitations the gold
didn't tag (*"this study did not evaluate…"*, *"the lack of prospective data collection could have
introduced bias"*). Same group → same span-based scheme → same ceiling.

## 8. Honest headline & caveats

**Best end-to-end funnel on real SAL papers: PubMedBERT-FT, A+B+C = 0.701** (broad Stage C) /
**0.568** (SAL-criteria). Stage A contributes a *measured* +0.15 F1 (not a proxy); Stage C adds +0.10.

**Caveats:**
- **Not the same yardstick as Lan's 0.821** — that's a curated ~20.7% pool; ours is natural per-paper (~11%).
  Precision is prevalence-dependent.
- **SAL/Kilicoglu precision is a lower bound** — their span-based gold excludes consequence sentences the
  model correctly flags (documented in §5–6).
- Stage C trades recall for precision; config is corpus-sensitive (±1 context helps; SAL-criteria is stricter).

**Open thread:** span/limitation-level scoring (credit a kept sentence if it falls in the same limitation
span as a gold positive) would give the broad judge's recall *and* honest precision — the clean
reconciliation. Not yet implemented.